### Set up data for agentic app to act as a data analyst for the Marketing project data

This project is a **Databricks-native application** and requires a Databricks workspace to run. The code will not execute outside of a Databricks environment without significant refactoring of these integrations. To view the logic, please read the `.py`/`.ipynb` files directly.

#### Create catalog/schema/volume

In [0]:
# set variables for catalog, schema, volume names 
catalog_name = "analytics"
schema_name = "marketing"
volume_name = "data_sources"

# create catalog, schema, volume if they don't exist already
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")


DataFrame[]

#### Next step in Databricks UI: 
Uploaded `data/customer_behaviour_summary.csv` to `analytics.marketing.data_sources`.  

This is exactly the same CSV (although renamed) as `marketing_rfm.csv` created in notebook`marketing_rfm.ipynb` in the Marketing project of this portfolio.  However, note that there is one new transformation, **field `Income_category`** before writing to the Delta table.

#### Now create the table with the transform:

In [0]:
# create table(s) from uploaded file(s), add Income_category field
from pyspark.sql import functions as F

def create_table(file_name, transform_fn=None):
    volume_file_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/{file_name}.csv"
    table_name = f"{catalog_name}.{schema_name}.{file_name}"

    df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .load(volume_file_path) 

    if transform_fn:
        df=transform_fn(df)
    spark.sql(f"DROP TABLE IF EXISTS {table_name}")

    df.write.mode("overwrite").format("delta").saveAsTable(table_name)

def add_income_category(df):
    quantiles = df.approxQuantile("Income", [0.25, 0.75], 0.01)
    q1, q3 = quantiles
    
    return df.withColumn("Income_category",
        F.when(F.col("Income") <= q1, "Low")
         .when(F.col("Income") <= q3, "Medium")
         .otherwise("High"))
    
create_table("customer_behaviour_summary", transform_fn=add_income_category)